# Unit 4 hands-on: fine-tune a music genre classifier (GTZAN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadDaleen/Audio-Course/blob/main/units/unit4_music_genre_classifier/colab_handson.ipynb)

This notebook walks you through the Hugging Face Audio Course **Unit 4 hands-on** from start to finish.

**What you'll do:** take a model that already understands speech/audio (`DistilHuBERT`) and *fine-tune*
it on the **GTZAN** dataset so it can label a music clip with one of 10 genres (blues, classical,
country, disco, hiphop, jazz, metal, pop, reggae, rock). Then you'll push your trained model to the
Hugging Face Hub.

**What is "fine-tuning"?** Instead of training a model from scratch (which needs huge data and
compute), we start from a pre-trained model that already learned general audio features, and we only
teach it the new task (genre labels) using a small dataset. It's fast and works well.

**Your goal:** reach **at least 87% accuracy** on the test set (the course baseline is 83%) and
**push the model to the Hub** — that is what makes the exercise count toward your certificate.

**Before you run anything:**
1. Set the runtime to a GPU: **Runtime → Change runtime type → T4 GPU → Save**. Training on CPU would
   take many hours; on a T4 it is about an hour.
2. Run the cells top to bottom (Shift+Enter on each, or **Runtime → Run all**).

## Step 1 — Install the libraries

We pin specific versions so this notebook keeps working:

- **`datasets==3.6.0`** — GTZAN is loaded by a small Python *script* hosted with the dataset. Version
  4 of `datasets` removed support for script datasets, so GTZAN would fail there. 3.x is required.
- **`transformers>=4.46`** — needed for the current training API used below.
- `evaluate` (accuracy metric), `accelerate` (runs training on the GPU), `soundfile`/`librosa`
  (decode the audio files), `tensorboard` (training charts).

We do **not** install PyTorch — Colab already ships a GPU build of it.

In [ ]:
!pip install -q "transformers>=4.46,<5" "datasets==3.6.0" "evaluate>=0.4" \
                "accelerate>=0.30" "soundfile>=0.12.1" "librosa>=0.10" tensorboard

> If a later cell raises a strange import error, do **Runtime → Restart session**, then run again
> from Step 2 (you don't need to re-run Step 1 — the packages are already installed).

## Step 2 — Check the GPU and log in to Hugging Face

The first line confirms a GPU is active. The login stores a token so the notebook can upload your
model at the end.

**Use a WRITE token — this is the #1 thing people get wrong.** A read-only token cannot create your
model repository and you'll get a `403 Forbidden` error later. Create one here:
**https://huggingface.co/settings/tokens → Create new token → type "Write" → Create → copy it.**

Run the cell, then paste the **write** token into the box that appears.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(),
      "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Runtime -> T4 GPU!")

from huggingface_hub import notebook_login
notebook_login()   # paste a WRITE token

Now confirm the login worked and the token can actually write. You want your own username and a role
of `write` (or a fine-grained token allowed to create repos):

In [ ]:
from huggingface_hub import whoami
info = whoami()
print("logged in as:", info["name"])
print("token role  :", info.get("auth", {}).get("accessToken", {}).get("role", "(fine-grained)"))

## Step 3 — Load the GTZAN dataset

GTZAN is **1000 music clips, 30 seconds each, 100 per genre** across 10 genres. We:

- load it (the first time downloads ~1.2 GB),
- split off **10% as a test set** to measure accuracy on clips the model never trained on,
- **resample to 16 kHz**, because that's the sample rate DistilHuBERT expects.

The download comes from an academic server that is occasionally slow or drops the connection — if the
cell errors, **just run it again**.

In [ ]:
from datasets import load_dataset, Audio

# (The course's load_dataset("marsyas/gtzan", "all") is outdated: there is no "all" config.)
gtzan = load_dataset("marsyas/gtzan", trust_remote_code=True)
gtzan = gtzan["train"].train_test_split(seed=42, shuffle=True, test_size=0.1)
gtzan = gtzan.cast_column("audio", Audio(sampling_rate=16_000))

print(gtzan)
print("genres:", gtzan["train"].features["genre"].names)

## Step 4 — Turn audio into model inputs (the feature extractor)

A model can't read a raw `.wav` file directly. The **feature extractor** prepares each clip:

- **normalizes** it (so loud and quiet clips are on the same scale),
- **pads or truncates** every clip to exactly 30 seconds, so all inputs are the same length.

`.map(...)` runs this over the whole dataset. We then drop the big raw-audio column (we only need the
extracted features) and rename the `genre` column to `label`, which is the name the trainer looks for.

In [ ]:
from transformers import AutoFeatureExtractor

model_id = "ntu-spml/distilhubert"
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True, return_attention_mask=True
)
sampling_rate = feature_extractor.sampling_rate   # 16000
max_duration = 30.0

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    return feature_extractor(
        audio_arrays,
        sampling_rate=sampling_rate,
        max_length=int(sampling_rate * max_duration),
        truncation=True,
        return_attention_mask=True,
    )

gtzan_encoded = gtzan.map(
    preprocess_function,
    remove_columns=["audio", "file"],   # keep only the extracted features
    batched=True,
    batch_size=100,
    num_proc=1,
)
gtzan_encoded = gtzan_encoded.rename_column("genre", "label")
gtzan_encoded

## Step 5 — Load the model

We load DistilHuBERT and attach a brand-new **classification head** with 10 outputs (one per genre).
The body of the model already understands audio; only this head needs training.

Passing `id2label`/`label2id` stores the genre names inside the model, so your uploaded model shows
labels like `rock` instead of `LABEL_7`.

In [ ]:
from transformers import AutoModelForAudioClassification

id2label_fn = gtzan["train"].features["genre"].int2str
id2label = {str(i): id2label_fn(i) for i in range(10)}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(id2label),
    label2id=label2id,
    id2label=id2label,
)

> You'll see a warning that some weights (`classifier`, `projector`) are "newly initialized". That is
> **expected and correct** — those are the new head we're about to train.

## Step 6 — Define the accuracy metric

**Accuracy** = how many test clips the model labels correctly, out of 100. This is the exact number
the hands-on is graded on (you want ≥ 0.87). The trainer will call this after every epoch.

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)   # pick the highest-scoring genre
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

## Step 7 — Training configuration

This cell sets the training "knobs". In plain terms:

- **`num_train_epochs=20`** — how many times the model sees the whole training set. The course used 10
  and got 83%; we use 20 to comfortably clear 87%.
- **`learning_rate=5e-5`** — how big each learning step is. Too high = unstable, too low = slow. This
  is a good default for fine-tuning.
- **`per_device_train_batch_size=8`** — how many clips the GPU processes at once. Lower this to 4 if
  you ever hit an "out of memory" error.
- **`warmup_ratio=0.1`** — start with a tiny learning rate and ramp up over the first 10% of steps;
  this stabilizes early training.
- **`fp16=True`** — use 16-bit math on the GPU: faster and uses less memory. (GPU only.)
- **`eval_strategy`/`save_strategy="epoch"`** — measure accuracy and save a checkpoint after every epoch.
- **`load_best_model_at_end=True`** with **`metric_for_best_model="accuracy"`** — at the end, keep the
  single best-scoring epoch, not necessarily the last one. This protects you if late epochs overfit.
- **`push_to_hub=True`** — upload checkpoints/model to your Hub account.
- **`report_to="tensorboard"`** — log training curves (and avoid Colab prompting for other loggers).

In [ ]:
from transformers import TrainingArguments, Trainer

model_name = model_id.split("/")[-1]   # "distilhubert"

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-gtzan",   # also becomes your Hub repo name
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    push_to_hub=True,
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=gtzan_encoded["train"],
    eval_dataset=gtzan_encoded["test"],
    processing_class=feature_extractor,   # current API (older tutorials used tokenizer=)
    compute_metrics=compute_metrics,
)

> **If this cell raises `403 Forbidden ... create`**, your token is read-only. Re-run Step 2 with a
> **Write** token, then run this cell again. (The repo is created here, the moment `push_to_hub=True`
> is set — before any training — so you lose no time fixing it.)

## Step 8 — Train (this is the ~1 hour step)

Run it and watch the table: after each epoch you'll see `Validation Loss` and `Accuracy`. You want
`Accuracy` to climb past **0.87**. Keep the Colab tab open so the runtime doesn't disconnect.

In [ ]:
trainer.train()

## Step 9 — Confirm your best accuracy

Because of `load_best_model_at_end`, `trainer` now holds the best checkpoint. Check that
`eval_accuracy` is at least **0.87**.

In [ ]:
results = trainer.evaluate()
print(results)

## Step 10 — Push to the Hub (this is what counts for the certificate)

This uploads your model plus a model card. The `kwargs` add tags (the GTZAN dataset, the
audio-classification task) so the course recognizes your submission.

In [ ]:
kwargs = {
    "dataset_tags": "marsyas/gtzan",
    "dataset": "GTZAN",
    "model_name": f"{model_name}-finetuned-gtzan",
    "finetuned_from": model_id,
    "tasks": "audio-classification",
}
trainer.push_to_hub(**kwargs)

Your model is now at `https://huggingface.co/<your-username>/distilhubert-finetuned-gtzan`. 🎉

## Step 11 (optional) — Try your model

Load your uploaded model as a ready-to-use pipeline and classify a test clip. Replace
`<your-username>` with your Hub username.

In [ ]:
from transformers import pipeline

repo = f"<your-username>/{model_name}-finetuned-gtzan"
classifier = pipeline("audio-classification", model=repo)

sample = gtzan["test"][0]
true_genre = gtzan["test"].features["genre"].int2str(sample["genre"])
print("true genre:", true_genre)
classifier(sample["audio"]["array"])

## Troubleshooting

- **`403 Forbidden ... create`** — your token is read-only. Make a **Write** token at
  https://huggingface.co/settings/tokens, re-run Step 2, then continue.
- **"NO GPU" printed in Step 2** — set **Runtime → Change runtime type → T4 GPU** and re-run.
- **`CUDA out of memory`** — lower `per_device_train_batch_size` to 4 in Step 7 and re-run.
- **GTZAN download fails / hangs** — the host is flaky; just re-run Step 3.
- **Accuracy stuck below 87%** — GTZAN's test set is only ~100 clips, so accuracy is noisy. Try
  `num_train_epochs=30`, change the `seed` in Step 3, or swap `model_id` in Step 4/5 to a stronger
  base model such as `"facebook/wav2vec2-base"` (the course explicitly allows a different model).
- **Runtime disconnected mid-training** — Colab free tier disconnects if idle; keep the tab open and
  active, or use a shorter `num_train_epochs`.